<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 02 — N-gram Language Models

**Course:** Natural Language Processing  
**Session:** 2 / 8  
**Duration:** ~2h30 (TP portion)

---

## Objectives

By the end of this session you will be able to:

- Build an n-gram language model from scratch using MLE
- Apply Laplace smoothing and explain why it is necessary
- Compute perplexity and use it to compare models
- Generate text by sampling from a trained n-gram LM
- Relate perplexity to cross-entropy

---

## Roadmap

| Part | Task | Deliverable |
|------|------|-------------|
| 1 | Setup & corpus statistics | Vocabulary size, bigram count |
| 2 | `NgramLM` class | `fit`, `log_prob`, `perplexity` |
| 3 | Smoothing | Laplace-smoothed PPL vs unsmoothed |
| 4 | Evaluation across $n$ and classes | PPL comparison table |
| 5 | Text generation | Class-conditioned sentences |

Each part ends with a **📝 Your analysis** cell.  
**Homework** (ex 5.5): Naive Bayes from scratch — due before Session 3.

---

> **Note:** this notebook picks up from TP01.  
> If you do not have `train_df`, `val_df`, `test_df`, `meta` in memory, re-run the data loading cell from TP01 or use the snippet in Part 1 below.

---
## Part 0 — Imports & data loading

In [ ]:
from __future__ import annotations

import re
import string
import random
import math
from collections import defaultdict, Counter
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from datasets import load_dataset

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── AG News dataset (same split as TP01) ──────────────────────────────────────
DATASET_NAME = "ag_news"
LABEL_NAMES  = ["World", "Sports", "Business", "Sci/Tech"]

raw       = load_dataset(DATASET_NAME)
train_raw = raw["train"].to_pandas()
test_raw  = raw["test"].to_pandas()

# Reproduce the TP01 train/val split (80/20, stratified)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    train_raw, test_size=0.2, random_state=SEED, stratify=train_raw["label"]
)
test_df = test_raw.copy()
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

meta = {
    "label_names": LABEL_NAMES,
    "num_classes":  len(LABEL_NAMES),
}

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(3)

In [ ]:
# ── Preprocessing pipeline from TP01 ─────────────────────────────────────────
# (copy the full pipeline function you validated in TP01)

def preprocess_text(text: str) -> str:
    """Normalise raw text: lowercase, strip HTML, remove punctuation, collapse spaces.

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string.
    """
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r" +", " ", text)
    return text.strip()


# Apply to all splits
for df in (train_df, val_df, test_df):
    df["text_clean"] = df["text"].map(preprocess_text)

print("Preprocessing done.")
print(train_df[["text", "text_clean"]].head(2).to_string())

---
## Part 1 — Corpus statistics

Before building a language model, understand the corpus at the token level.
The statistics here feed directly into Part 2 (you will reuse `tokenize` and the
vocabulary).

In [ ]:
# ── 1.1  Tokeniser ────────────────────────────────────────────────────────────

BOS = "<s>"   # beginning-of-sentence token
EOS = "</s>"  # end-of-sentence token
UNK = "<unk>" # out-of-vocabulary token


def tokenize(text: str) -> list[str]:
    """Whitespace-split a preprocessed string into a list of tokens.

    Parameters
    ----------
    text : str
        Preprocessed (lowercase, no punctuation) text.

    Returns
    -------
    list[str]
        List of token strings. Empty string returns empty list.
    """
    return text.split()


# YOUR CODE HERE
# 1. Tokenise all documents in train_df["text_clean"].
# 2. Build a Counter over all tokens (unigrams).
# 3. Print: total tokens, vocabulary size, top-10 tokens.
raise NotImplementedError

In [ ]:
def build_vocabulary(
    token_counter: Counter,
    min_freq: int = 2,
) -> set[str]:
    """Build a vocabulary by frequency thresholding.

    Tokens with count < min_freq are excluded from the vocabulary.
    Special tokens BOS, EOS, UNK are always included.

    Parameters
    ----------
    token_counter : Counter
        Token frequency counter built from the training corpus.
    min_freq : int
        Minimum count for a token to be included in the vocabulary.

    Returns
    -------
    set[str]
        Vocabulary set including special tokens.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# YOUR CODE HERE
# Call build_vocabulary. Print vocabulary size before and after thresholding.
raise NotImplementedError

### 📝 Your analysis — Part 1

**Q1.** What fraction of the vocabulary (by type count) is covered by tokens with frequency $\geq 2$? What fraction of total token occurrences (by token count) is covered?

**Q2.** How does the vocabulary size compare to the BPE vocabulary from TP01? What are the trade-offs?

**Q3.** Why do we add boundary tokens `<s>` and `</s>` for language modelling, but not for classification?

*Write your answers in this cell.*

---
## Part 2 — The `NgramLM` class

Implement a general n-gram language model. The class must support $n = 1, 2, 3$ (unigram,
bigram, trigram) and Laplace smoothing.

**Design decisions you must follow:**

- Store n-gram counts as a `defaultdict` of `Counter`s keyed by the $(n-1)$-gram context.
- Store the vocabulary and special tokens as instance attributes.
- Work in log-space internally: `log_prob` returns $\log P$ (natural log), never raw probability.
- Unknown words at inference time map to `UNK`.

In [ ]:
class NgramLM:
    """N-gram language model with Laplace smoothing.

    Estimates $P(w_t | w_{t-n+1}, ..., w_{t-1})$ from a training corpus
    using maximum likelihood estimation with optional Laplace (add-alpha) smoothing.

    Attributes
    ----------
    n : int
        Order of the language model (1 = unigram, 2 = bigram, ...).
    alpha : float
        Laplace smoothing parameter. 0 = no smoothing, 1 = add-one.
    vocab : set[str]
        Vocabulary set. Set after calling ``fit``.
    counts : defaultdict[tuple[str, ...], Counter]
        N-gram counts: context (n-1)-gram -> Counter over next tokens.
    context_totals : Counter
        Total count for each (n-1)-gram context, used as denominator.
    """

    def __init__(self, n: int = 2, alpha: float = 1.0) -> None:
        """Initialise the NgramLM.

        Parameters
        ----------
        n : int
            Order of the n-gram model.
        alpha : float
            Laplace smoothing pseudocount. Use 0.0 to disable smoothing.
        """
        if n < 1:
            raise ValueError(f"n must be >= 1, got {n}")
        self.n     = n
        self.alpha = alpha
        self.vocab: set[str] = set()
        self.counts: defaultdict = defaultdict(Counter)
        self.context_totals: Counter = Counter()

    # ------------------------------------------------------------------
    def _add_boundaries(self, tokens: list[str]) -> list[str]:
        """Wrap a token sequence with BOS/EOS boundary tokens.

        Prepends (n-1) BOS tokens and one EOS token so that every
        position in the sequence has a full context of length n-1.

        Parameters
        ----------
        tokens : list[str]
            Raw token sequence for one sentence.

        Returns
        -------
        list[str]
            Padded token sequence with boundary tokens.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    # ------------------------------------------------------------------
    def _unk_replace(self, tokens: list[str]) -> list[str]:
        """Replace out-of-vocabulary tokens with the UNK special token.

        Parameters
        ----------
        tokens : list[str]
            Token sequence (boundaries already added or not).

        Returns
        -------
        list[str]
            Sequence with OOV tokens replaced by UNK.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    # ------------------------------------------------------------------
    def fit(self, corpus: Iterable[list[str]], vocab: set[str]) -> "NgramLM":
        """Fit the language model on a tokenised corpus.

        Counts all n-grams in the corpus and stores context totals
        for efficient probability estimation.

        Parameters
        ----------
        corpus : Iterable[list[str]]
            Iterable of tokenised sentences (lists of strings).
            Sentences must NOT already have boundary tokens.
        vocab : set[str]
            Pre-built vocabulary (from ``build_vocabulary``).
            Tokens outside this set are replaced with UNK.

        Returns
        -------
        NgramLM
            self, to allow method chaining.
        """
        self.vocab = vocab
        # YOUR CODE HERE
        # For each sentence:
        #   1. Apply _unk_replace.
        #   2. Apply _add_boundaries.
        #   3. Slide a window of size n and count each n-gram.
        #      Key = context tuple (first n-1 tokens).
        #      Value counted = last token.
        # After the loop, compute context_totals from counts.
        raise NotImplementedError

    # ------------------------------------------------------------------
    def log_prob(
        self,
        token: str,
        context: tuple[str, ...],
    ) -> float:
        """Return the log-probability of a token given its context.

        Uses Laplace smoothing with parameter alpha.
        Unknown tokens and unknown contexts are handled via UNK
        and the smoothing floor respectively.

        Parameters
        ----------
        token : str
            The token whose probability is estimated.
        context : tuple[str, ...]
            The (n-1) preceding tokens. Must have length n-1.
            For unigrams (n=1), pass an empty tuple ().

        Returns
        -------
        float
            Natural log of the smoothed probability. Always finite.

        Raises
        ------
        ValueError
            If context length does not match n-1.
        """
        if len(context) != self.n - 1:
            raise ValueError(
                f"Expected context length {self.n - 1}, got {len(context)}"
            )
        # YOUR CODE HERE
        # Replace OOV token and context tokens with UNK.
        # Numerator:   counts[context][token] + alpha
        # Denominator: context_totals[context] + alpha * |V|
        # Return math.log(numerator / denominator)
        raise NotImplementedError

    # ------------------------------------------------------------------
    def sentence_log_prob(self, tokens: list[str]) -> float:
        """Compute the log-probability of a tokenised sentence.

        Applies boundary tokens, replaces OOV with UNK, then sums
        log_prob over each position.

        Parameters
        ----------
        tokens : list[str]
            Tokenised sentence WITHOUT boundary tokens.

        Returns
        -------
        float
            Total log-probability of the sentence (sum over positions).
        """
        # YOUR CODE HERE
        # 1. Apply _unk_replace and _add_boundaries.
        # 2. Slide the window: context = padded[i:i+n-1], token = padded[i+n-1].
        # 3. Sum log_prob over all positions.
        raise NotImplementedError

    # ------------------------------------------------------------------
    def perplexity(self, corpus: Iterable[list[str]]) -> float:
        """Compute perplexity of the model on a tokenised corpus.

        Perplexity is defined as:

        .. math::
            \\text{PPL} = \\exp\\left(-\\frac{1}{N}\\sum_{t} \\log P(w_t | \\text{context})\\right)

        where N is the total number of predicted tokens (excluding BOS padding).

        Parameters
        ----------
        corpus : Iterable[list[str]]
            Iterable of tokenised sentences WITHOUT boundary tokens.

        Returns
        -------
        float
            Perplexity. Lower is better.
        """
        # YOUR CODE HERE
        # Accumulate total log-prob and total token count across all sentences.
        # Return math.exp(-total_log_prob / total_tokens)
        raise NotImplementedError

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
# Run these assertions after completing NgramLM.

# Build a tiny corpus
_tiny_vocab = build_vocabulary(Counter(["the", "cat", "sat", "the", "cat", "mat", "on"]), min_freq=1)
_tiny_corpus = [
    ["the", "cat", "sat"],
    ["the", "cat", "sat", "on", "the", "mat"],
]

# Bigram model
_lm = NgramLM(n=2, alpha=1.0).fit(_tiny_corpus, _tiny_vocab)

# log_prob must be <= 0
lp = _lm.log_prob("cat", ("the",))
assert lp <= 0, f"log_prob must be <= 0, got {lp}"

# log_prob of UNK must be finite
lp_unk = _lm.log_prob("invisible_word", ("the",))
assert math.isfinite(lp_unk), "log_prob of OOV word must be finite (Laplace smoothing)"

# perplexity must be > 1
ppl = _lm.perplexity(_tiny_corpus)
assert ppl > 1, f"PPL must be > 1, got {ppl}"

# sentence_log_prob must be negative
slp = _lm.sentence_log_prob(["the", "cat"])
assert slp < 0, f"sentence_log_prob must be < 0, got {slp}"

print("All sanity checks passed.")
print(f"  bigram PPL on tiny corpus: {ppl:.2f}")

### 📝 Your analysis — Part 2

**Q1.** Explain in your own words why `sentence_log_prob` sums log-probabilities rather than multiplying raw probabilities.

**Q2.** What happens to `log_prob` if `alpha = 0` and the n-gram context has never been seen? How does this affect perplexity computation?

**Q3.** For the unigram model (`n=1`), what is the context passed to `log_prob`? What does the model effectively compute?

*Write your answers in this cell.*

---
## Part 3 — Smoothing

Compare unsmoothed ($\alpha = 0$) and Laplace-smoothed ($\alpha = 1$) bigram models.
Demonstrate the zero-count problem concretely.

In [ ]:
# ── 3.1  Fit bigram models with and without smoothing ─────────────────────────

# YOUR CODE HERE
# 1. Build the training corpus: tokenise all texts in train_df["text_clean"].
# 2. Build the vocabulary with min_freq=2.
# 3. Fit two bigram models: alpha=0.0 and alpha=1.0.
raise NotImplementedError

In [ ]:
def demonstrate_zero_count(
    lm_no_smooth: NgramLM,
    lm_smooth: NgramLM,
    test_sentence: list[str],
) -> None:
    """Show the effect of a zero-count bigram on sentence log-probability.

    Prints the log-probability under both models for the given sentence,
    highlighting which bigram triggers the zero-count issue.

    Parameters
    ----------
    lm_no_smooth : NgramLM
        Bigram LM trained without Laplace smoothing (alpha=0).
    lm_smooth : NgramLM
        Bigram LM trained with Laplace smoothing (alpha=1).
    test_sentence : list[str]
        Tokenised sentence that contains at least one unseen bigram.
    """
    # YOUR CODE HERE
    # For each bigram in the sentence:
    #   - print log_prob under both models side by side
    #   - mark bigrams where lm_no_smooth returns -inf or math.log(0)
    raise NotImplementedError


# Pick a sentence from the val set that is likely to contain an unseen bigram
_test_sent = tokenize(val_df.iloc[0]["text_clean"])
demonstrate_zero_count(lm_bigram_raw, lm_bigram_smooth, _test_sent)

### 📝 Your analysis — Part 3

**Q1.** For the sentence you tested, how many bigrams were unseen? What fraction of the sentence does that represent?

**Q2.** Laplace smoothing transfers probability mass to unseen events. What is the cost? (Hint: look at the log-prob of a very common bigram under smoothed vs unsmoothed.)

**Q3.** (Open-ended) Kneser-Ney smoothing is the standard for n-gram LMs in production. Read the Wikipedia description. What key property does it have that Laplace does not?

*Write your answers in this cell.*

---
## Part 4 — Evaluation: PPL across $n$ and across classes

Train unigram, bigram, and trigram models on the full training set.
Evaluate perplexity on the test set, overall and per AG News class.

In [ ]:
def train_ngram_lm(
    train_df: pd.DataFrame,
    text_col: str,
    n: int,
    alpha: float,
    min_freq: int = 2,
) -> NgramLM:
    """Convenience wrapper: build vocabulary and fit an NgramLM in one call.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training dataframe with a preprocessed text column.
    text_col : str
        Column containing preprocessed text strings.
    n : int
        N-gram order.
    alpha : float
        Laplace smoothing parameter.
    min_freq : int
        Minimum token frequency for vocabulary inclusion.

    Returns
    -------
    NgramLM
        Fitted NgramLM instance.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# YOUR CODE HERE
# Train unigram, bigram, trigram models (all with alpha=1.0).
# Compute test-set perplexity for each. Store in a dict.
raise NotImplementedError

In [ ]:
def compute_per_class_ppl(
    lm: NgramLM,
    test_df: pd.DataFrame,
    text_col: str,
    label_col: str,
    label_names: list[str],
) -> pd.DataFrame:
    """Compute perplexity of an NgramLM separately for each class.

    Parameters
    ----------
    lm : NgramLM
        Fitted language model.
    test_df : pd.DataFrame
        Test dataframe with preprocessed text and labels.
    text_col : str
        Column containing preprocessed text.
    label_col : str
        Column containing integer class labels.
    label_names : list[str]
        Human-readable class names indexed by label integer.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ["class", "n_docs", "perplexity"],
        one row per class.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# YOUR CODE HERE
# Compute per-class PPL for the bigram model.
# Display the results as a styled DataFrame.
raise NotImplementedError

In [ ]:
# ── 4.3  PPL summary plot ──────────────────────────────────────────────────────

def plot_ppl_comparison(
    ppl_by_n: dict[int, float],
    title: str = "Perplexity vs n-gram order (AG News test set)",
) -> None:
    """Bar chart comparing perplexity across n-gram orders.

    Parameters
    ----------
    ppl_by_n : dict[int, float]
        Mapping from n-gram order to perplexity value.
    title : str
        Plot title.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# YOUR CODE HERE
# Call plot_ppl_comparison with your results.
raise NotImplementedError

### 📝 Your analysis — Part 4

**Q1.** Fill in this table with your experimental results:

| Model | Test PPL |
|-------|----------|
| Unigram | ??? |
| Bigram | ??? |
| Trigram | ??? |

**Q2.** Which AG News class has the lowest perplexity under the bigram model? Explain why this is expected, given what you know about each class's vocabulary from TP01.

**Q3.** (Open-ended) Why does perplexity decrease as $n$ increases, but only up to a point? What happens when $n$ is too large relative to the corpus size?

*Write your answers in this cell.*

---
## Part 5 — Text generation

Sample text from the trained bigram and trigram models.
Use temperature to control randomness.  
Generate class-conditioned text by training per-class models.

In [ ]:
def generate(
    lm: NgramLM,
    max_tokens: int = 30,
    temperature: float = 1.0,
    seed: int | None = None,
) -> str:
    """Generate a sentence by sampling from the n-gram LM.

    Starts from the BOS context and samples tokens one by one until
    EOS is produced or max_tokens is reached.

    Temperature scales the logits before softmax:
    - temperature < 1: sharpens the distribution (more deterministic)
    - temperature > 1: flattens the distribution (more random)

    Parameters
    ----------
    lm : NgramLM
        Fitted NgramLM.
    max_tokens : int
        Maximum number of tokens to generate (excluding BOS/EOS).
    temperature : float
        Sampling temperature. Must be > 0.
    seed : int or None
        Random seed for reproducibility. None = no seeding.

    Returns
    -------
    str
        Generated sentence as a space-joined string (no special tokens).
    """
    if temperature <= 0:
        raise ValueError(f"temperature must be > 0, got {temperature}")
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # YOUR CODE HERE
    # 1. Initialise context with (n-1) BOS tokens.
    # 2. At each step:
    #    a. Get all candidate tokens from lm.counts[context].
    #    b. Compute log-probs for all candidates.
    #    c. Apply temperature: log_probs /= temperature.
    #    d. Convert to probabilities with softmax.
    #    e. Sample one token using np.random.choice.
    #    f. Append to generated sequence. Update context.
    #    g. Stop if EOS is sampled.
    # 3. Return " ".join(generated_tokens).
    raise NotImplementedError


# Quick test
print(generate(lm_bigram_smooth, seed=SEED))

In [ ]:
# ── 5.2  Class-conditioned generation ─────────────────────────────────────────

# YOUR CODE HERE
# For each AG News class:
#   1. Filter train_df to documents with that class label.
#   2. Train a bigram LM on that subset.
#   3. Generate 3 sentences and print them.
raise NotImplementedError

In [ ]:
# ── 5.3  Temperature sweep ─────────────────────────────────────────────────────

def temperature_sweep(
    lm: NgramLM,
    temperatures: list[float],
    n_samples: int = 3,
    seed: int = SEED,
) -> None:
    """Print generated sentences at multiple temperature values.

    Parameters
    ----------
    lm : NgramLM
        Fitted NgramLM.
    temperatures : list[float]
        List of temperature values to sweep.
    n_samples : int
        Number of sentences to generate per temperature.
    seed : int
        Initial random seed.
    """
    # YOUR CODE HERE
    raise NotImplementedError


temperature_sweep(lm_bigram_smooth, temperatures=[0.5, 1.0, 2.0])

### 📝 Your analysis — Part 5

**Q1.** Compare 3 sentences generated by the bigram vs trigram model at temperature 1.0. Which produces more coherent text? Why?

**Q2.** Look at the class-conditioned output. Can you recognise which class each sentence comes from without reading the label? What tokens give it away?

**Q3.** (Open-ended) What is the fundamental limit of n-gram generation compared to a neural LM? Give a concrete example of something a trigram LM cannot generate correctly.

*Write your answers in this cell.*

---
## Part 5.5 — Homework: Naive Bayes from scratch ⭐

> **Due before Session 3.** Bring your macro F1 result.

Implement a multinomial Naive Bayes classifier **without** using `sklearn.naive_bayes`.  
You may use `sklearn` only for evaluation (`classification_report`, `f1_score`).

### What to implement

1. **`NaiveBayes.fit(X, y)`** — estimate log-priors and log-likelihoods with Laplace smoothing.
2. **`NaiveBayes.predict(X)`** — return predicted class labels using log-space argmax.
3. **`NaiveBayes.predict_log_proba(X)`** — return the (unnormalised) log-posterior matrix.

### Feature representation

Use a **bag-of-words** count matrix. You _may_ use `sklearn.feature_extraction.text.CountVectorizer` to build it.

### Evaluation

Report:
- Macro F1 on the AG News test set
- Per-class precision, recall, F1
- Comparison table with the majority baseline from TP01

### Hints

- Work in log-space throughout (avoid underflow).
- The vectoriser must be fit only on `train_df`.
- Add Laplace smoothing: $\hat{P}(w | c) = \frac{\text{count}(w, c) + 1}{\sum_{w'} \text{count}(w', c) + |V|}$.

In [ ]:
from scipy.sparse import spmatrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, f1_score


class NaiveBayes:
    """Multinomial Naive Bayes classifier with Laplace smoothing.

    Implements the generative classifier:
        P(y | x) \\propto P(y) * \\prod_i P(x_i | y)

    All computations are performed in log-space to avoid underflow.

    Attributes
    ----------
    alpha : float
        Laplace smoothing pseudocount.
    classes_ : np.ndarray
        Sorted array of unique class labels. Set after ``fit``.
    log_priors_ : np.ndarray
        Log-prior probabilities, shape (n_classes,). Set after ``fit``.
    log_likelihoods_ : np.ndarray
        Log-word likelihoods, shape (n_classes, n_features). Set after ``fit``.
    """

    def __init__(self, alpha: float = 1.0) -> None:
        """Initialise NaiveBayes.

        Parameters
        ----------
        alpha : float
            Laplace smoothing pseudocount added to all word counts.
        """
        self.alpha = alpha
        self.classes_: np.ndarray | None = None
        self.log_priors_: np.ndarray | None = None
        self.log_likelihoods_: np.ndarray | None = None

    def fit(self, X: spmatrix, y: np.ndarray) -> "NaiveBayes":
        """Estimate model parameters from a labelled training set.

        Computes log-prior $\\log P(y = c)$ and log-likelihood
        $\\log P(w | y = c)$ for all classes and vocabulary tokens.

        Parameters
        ----------
        X : sparse matrix, shape (n_samples, n_features)
            Bag-of-words count matrix.
        y : np.ndarray, shape (n_samples,)
            Integer class labels.

        Returns
        -------
        NaiveBayes
            self.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def predict_log_proba(self, X: spmatrix) -> np.ndarray:
        """Compute unnormalised log-posterior scores for each class.

        Returns log P(y=c) + sum_i log P(x_i | y=c) for each sample and class.

        Parameters
        ----------
        X : sparse matrix, shape (n_samples, n_features)
            Bag-of-words count matrix.

        Returns
        -------
        np.ndarray, shape (n_samples, n_classes)
            Unnormalised log-posterior matrix.
        """
        # YOUR CODE HERE
        # log_posterior = X @ log_likelihoods_.T + log_priors_
        raise NotImplementedError

    def predict(self, X: spmatrix) -> np.ndarray:
        """Predict the most probable class for each sample.

        Parameters
        ----------
        X : sparse matrix, shape (n_samples, n_features)
            Bag-of-words count matrix.

        Returns
        -------
        np.ndarray, shape (n_samples,)
            Predicted class labels.
        """
        # YOUR CODE HERE
        raise NotImplementedError


# ── YOUR CODE — train and evaluate ───────────────────────────────────────────
# 1. Fit a CountVectorizer on train_df["text_clean"].
# 2. Transform train, val, test.
# 3. Fit NaiveBayes on train.
# 4. Predict on test. Print classification_report.
# 5. Build a comparison DataFrame: majority baseline vs NB.
raise NotImplementedError

---
## Summary

Run the cell below to print your final PPL table and NB result (if done).

In [ ]:
print(f"Dataset : {DATASET_NAME}")
print(f"Train   : {len(train_df):,}  Val : {len(val_df):,}  Test : {len(test_df):,}")
print()
print("Perplexity summary:")
# Replace with your actual results
for n, ppl in ppl_by_n.items():
    print(f"  {n}-gram : PPL = {ppl:.1f}")
print()
print("Homework NB (if done):")
print(f"  Macro F1 = ???")

---
## What's next?

In **TP 03** you will train Naive Bayes and Logistic Regression classifiers on AG News,
compare their performance, and interpret their weights.
The class fingerprinting from TP01 will tell you exactly which tokens these models rely on.

Before moving on, save reusable pieces such as the `NgramLM` class, AG News loading/splitting code, and your perplexity table to a small `.py` module or data file that future notebooks can import. This is more robust than relying on this notebook session staying open.